In [1]:
import requests
import numpy as np
import json
import time
import re
from datetime import datetime

1)The LLMAdvisor class connects to a local Ollama LLM server and checks if a language model (like llama3.2:1b) is available to generate advice.

2)It collects system metrics such as accuracy, precision, exploration rate (epsilon), rewards, and active learning statistics from the IoT anomaly detection system.

3)These metrics are converted into a prompt, which is sent to the LLM to request structured advice in JSON format.

4)The response from the LLM is parsed, validated, and cached, ensuring the advice values (confidence, epsilon adjustment, strategy) are within valid limits.

5)Finally, the system prints the advice and tracks statistics, helping guide reinforcement learning exploration and active learning strategies during training.

In [ ]:
class LLMAdvisor:
#Initializes the LLM advisor by setting the model name, temperature, and API URL.It also checks if Ollama is running and warms up the model if available.
    def __init__(self, model="llama3.2:1b", temperature=0.3, timeout=60):
        self.model = model
        self.temperature = temperature
        self.timeout = timeout
        self.url = "http://localhost:11434"
        self.available = self._check_connection()

        self.advice_count = 0
        self.total_response_time = 0
        self.cache = {}
        self.last_advice = None

        if self.available:
            self._warm_model()

#Checks whether the Ollama server is running on localhost:11434. It also retrieves the list of available models and selects one to use

    def _check_connection(self):
        try:
            response = requests.get(f"{self.url}/api/tags", timeout=5)
            if response.status_code == 200:
                models = response.json().get('models', [])
                model_names = [m['name'] for m in models]
                print(f" Connected to Ollama")
                print(f"  Available models: {model_names}")
                if self.model in model_names:
                    print(f"  Using model: {self.model}")
                    return True
                elif model_names:
                    self.model = model_names[0]
                    print(f"  Using fallback model: {self.model}")
                    return True
            return False
        except Exception as e:
            print(f"⚠️ Cannot connect to Ollama: {e}")
            print("  Make sure Ollama is running (run 'ollama serve')")
            return False

#Sends a small prompt (“Hello”) to the LLM to load the model into memory. This reduces delay when the first real request is made.
    def _warm_model(self):
        try:
            print("Warming up LLM model...")
            requests.post(
                f"{self.url}/api/generate",
                json={
                    "model": self.model,
                    "prompt": "Hello",
                    "stream": False,
                    "options": {
                        "num_predict": 5
                    }
                },
                timeout=30
            )
            print(" Model warmed up")
        except:
            pass

  # Main function that sends system metrics (accuracy, epsilon, etc.) to the LLM. It receives AI advice in JSON format and returns it after parsing and validation
    def get_advice(self, context):
        if not self.available:
            return self._default_advice()

        cache_key = f"{context.get('accuracy',0):.3f}_{context.get('epsilon',0):.3f}_{context.get('queries_used',0)}"

        if cache_key in self.cache:
            advice = self.cache[cache_key].copy()
            advice['from_cache'] = True
            return advice

        prompt = self._create_prompt(context)

        try:
            start_time = time.time()

            response = requests.post(
                f"{self.url}/api/generate",
                json={
                    "model": self.model,
                    "prompt": prompt,
                    "stream": False,
                    "temperature": self.temperature,
                    "options": {
                        "num_predict": 200,
                        "temperature": self.temperature,
                        "top_k": 40,
                        "top_p": 0.9
                    }
                },
                timeout=self.timeout
            )

            elapsed = time.time() - start_time
            self.total_response_time += elapsed

            if response.status_code == 200:
                result = response.json()
                text = result.get('response', '')

                advice = self._parse_response(text)

                if advice:
                    advice['response_time'] = round(elapsed, 2)
                    self.advice_count += 1
                    self.last_advice = advice
                    self.cache[cache_key] = advice
                    self._print_advice(advice, elapsed)
                    return advice

            return self._default_advice()

        except Exception as e:
            print(f" LLM error: {e}")
            return self._default_advice()

  #Builds the prompt text using system metrics like accuracy, recall, and reward. This prompt instructs the LLM to generate structured JSON advice
    def _create_prompt(self, context):
        accuracy = context.get('accuracy', 0) * 100
        precision = context.get('precision', 0) * 100
        recall = context.get('recall', 0) * 100
        f1 = context.get('f1_score', 0)
        epsilon = context.get('epsilon', 1.0)
        queries_used = context.get('queries_used', 0)
        budget = context.get('budget', 50)
        pseudo_labels = context.get('pseudo_labels', 0)
        alert_rate = context.get('alert_rate', 0) * 100
        avg_reward = context.get('avg_reward', 0)
        steps = context.get('steps', 0)
        vae_score = context.get('vae_score', 0)

        prompt = f"""You are an expert AI advisor for an IoT anomaly detection system with active learning.

Current System State:
- Accuracy: {accuracy:.1f}%
- Precision: {precision:.1f}%
- Recall: {recall:.1f}%
- F1 Score: {f1:.3f}
- Exploration Rate (epsilon): {epsilon:.3f}
- Alert Rate: {alert_rate:.1f}%
- Average Reward: {avg_reward:.2f}
- Training Steps: {steps}
- VAE Anomaly Score: {vae_score:.3f}

Active Learning Status:
- Queries Used: {queries_used}/{budget}
- Pseudo-labels Generated: {pseudo_labels}

Based on this state, provide expert advice in JSON format with the following fields:

1. analysis: Brief 1-sentence analysis of current situation
2. epsilon_adjustment: How much to adjust exploration (-0.1 to +0.1)
3. focus: Primary focus area ("exploration", "exploitation", "active_learning", or "stability")
4. al_strategy: Active learning strategy suggestion ("more_queries", "more_propagation", "balance", or "current")
5. confidence: Confidence in advice (0.0 to 1.0)
6. suggestion: Specific actionable suggestion

Return ONLY valid JSON in this format:
{{
    "analysis": "Brief analysis here",
    "epsilon_adjustment": 0.0,
    "focus": "exploration",
    "al_strategy": "balance",
    "confidence": 0.8,
    "suggestion": "Specific suggestion here"
}}"""

        return prompt
  #Extracts JSON data from the LLM response using regular expressions. Then converts the JSON string into a Python dictionary.
    def _parse_response(self, text):
        try:
            json_match = re.search(r'\{.*\}', text, re.DOTALL)
            if json_match:
                advice = json.loads(json_match.group())
                return self._validate_advice(advice)
        except:
            pass
        return None
#Checks whether the LLM output contains all required fields. It also ensures values like confidence and epsilon adjustment are within valid ranges.
    def _validate_advice(self, advice):
        default = self._default_advice()

        required = ['analysis', 'epsilon_adjustment', 'focus', 'al_strategy', 'confidence', 'suggestion']
        for key in required:
            if key not in advice:
                advice[key] = default[key]

        try:
            advice['confidence'] = float(advice['confidence'])
            advice['confidence'] = np.clip(advice['confidence'], 0, 1)
        except:
            advice['confidence'] = default['confidence']

        try:
            advice['epsilon_adjustment'] = float(advice['epsilon_adjustment'])
            advice['epsilon_adjustment'] = np.clip(advice['epsilon_adjustment'], -0.1, 0.1)
        except:
            advice['epsilon_adjustment'] = default['epsilon_adjustment']

        valid_focus = ['exploration', 'exploitation', 'active_learning', 'stability']
        if advice['focus'] not in valid_focus:
            advice['focus'] = default['focus']

        valid_al = ['more_queries', 'more_propagation', 'balance', 'current']
        if advice['al_strategy'] not in valid_al:
            advice['al_strategy'] = default['al_strategy']

        return advice
  #Returns a safe fallback advice when the LLM fails or is unavailable. This prevents the system from crashing during training.
    def _default_advice(self):
        return {
            'analysis': 'Continue current training strategy',
            'epsilon_adjustment': 0.0,
            'focus': 'stability',
            'al_strategy': 'current',
            'confidence': 0.5,
            'suggestion': 'Maintain current approach and monitor metrics'
        }
    #Prints the LLM advice in a readable format in the console. It shows analysis, epsilon adjustment, strategy, confidence, and response time.
    def _print_advice(self, advice, elapsed):
        print(f"\n{'🤖'*30}")
        print(f"🤖 LLM ADVICE #{self.advice_count}")
        print(f"{'🤖'*30}")
        print(f"⏱  Response: {elapsed:.1f}s")
        print(f"📊 Analysis: {advice['analysis']}")
        print(f"📈 Epsilon Adjustment: {advice['epsilon_adjustment']:+.3f}")
        print(f"🎯 Focus: {advice['focus']}")
        print(f"🎲 AL Strategy: {advice['al_strategy']}")
        print(f"✨ Confidence: {advice['confidence']:.1%}")
        print(f"💡 Suggestion: {advice['suggestion']}")
        if advice.get('from_cache'):
            print(f"📦 (from cache)")
        print(f"{'🤖'*30}\n")

    def get_stats(self):
        return {
            'advice_count': self.advice_count,
            'cache_size': len(self.cache),
            'avg_response_time': self.total_response_time / max(1, self.advice_count),
            'model': self.model,
            'available': self.available
        }
